In [1]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import torch
from tqdm import tqdm
from torch.nn.functional import pad

In [2]:
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
from transformers import LlamaForCausalLM, LlamaTokenizerFast

import argparse
import pickle
import os
import json
import pandas as pd

model_id = "gpt2"
device = "cuda:1"
perplexity_func = "p1"

if "gpt2" in model_id:
    model = GPT2LMHeadModel.from_pretrained(model_id).to(device)
    tokenizer = GPT2TokenizerFast.from_pretrained(model_id)
    start_of_sentence=" "
print(f"Loaded model: {model_id}")

/u/sebono/miniconda3/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded model: gpt2


In [3]:
model_id = "gpt2"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model : GPT2LMHeadModel = GPT2LMHeadModel.from_pretrained(model_id).to(device)
tokenizer : GPT2TokenizerFast = GPT2TokenizerFast.from_pretrained(model_id)

### Plotting

In [4]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats
import plotly
import plotly.io as pio
pio.renderers.default = 'iframe'
import plotly.express as px
plotly.offline.init_notebook_mode(connected=True)
import seaborn as sns

cmp = 'algae'
def correlation_heatmap(y_cols, x_cols, full_data):
    '''
    Uses scipy.stats.spearmanr function
    Params:
    y_cols, x_cols: sets of column titles (strings)
    full_data: pandas dataframe that includes all columns listed in y_cols, x_cols
    Returns:
    corr: Spearman correlation coefficient matrix (y_cols = rows, x_cols = cols of matrix)
    fig_corr: annotated plotly heatmap of coefficients
    p: Spearman p-value matrix
    fig_p: annotated plotly heatmap of p-values
    '''
    cols = y_cols+x_cols
    all_correlations = scipy.stats.spearmanr(full_data[cols], nan_policy='omit')
    corr = all_correlations.correlation[:len(y_cols), -len(x_cols):]
    corr = pd.DataFrame(corr)
    corr.columns = x_cols
    corr.index = y_cols

    p = all_correlations.pvalue[:len(y_cols), -len(x_cols):]
    p = pd.DataFrame(p)
    p.columns = x_cols
    p.index = y_cols
    
    fig_corr = px.imshow(corr, text_auto=True, aspect='auto', color_continuous_scale='agsunset')
    fig_r2 = px.imshow(corr**2, text_auto=True, aspect='auto', color_continuous_scale='agsunset')
    fig_p = px.imshow(p, text_auto=True, aspect='auto', color_continuous_scale='gray_r')

    return corr, fig_corr, p, fig_p, fig_r2

In [5]:
def compute_dominance_per_spk(perplexity):
    prev_idx_pp = 0
    tokens_ids_per_sentence = np.cumsum([t.size(0) for t in token_list])
    dialog = [tokenizer.decode(token, skip_special_tokens=True) for token in token_list]
    all_values = []
    ppls_p_spk={}
    for idx, match in enumerate(matches):
        if match not in ppls_p_spk:
            ppls_p_spk[match] = []
        for px in range(len(token_list[idx])):
            fin_idx = px + np.sum([len(l) for l in token_list[:idx]], dtype=int)
            ppls_p_spk[match].append(perplexity[fin_idx])
    return ppls_p_spk

In [6]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

def rolling_kde_heatmap_with_turns(
    ppls,
    token_list,
    window_size=50,
    step=10,
    bandwidth=0.5,
    vmin=None,
    vmax=None,
    title="[SPK]",
):
    """
    Plots a rolling KDE heatmap over token-level perplexities and marks turn boundaries.
    Optionally returns the KDE densities array.

    Args:
        ppls (List[float]): Flattened token-level perplexities.
        token_list (List[Tensor]): List of per-turn token tensors.
        window_size (int): Window size for KDE.
        step (int): Step size for moving window.
        bandwidth (float): Bandwidth for Gaussian KDE.
        vmin, vmax: Color limits for the heatmap.
        title (str): Title string.
        return_densities (bool): If True, return the densities matrix.

    Returns:
        If return_densities=True, returns the (n_windows x n_bins) KDE matrix.
    """
    ppls = np.array(ppls)
    xs = np.linspace(0, np.nanmax(ppls), 100)
    densities = []

    for i in range(0, len(ppls) - window_size, step):
        window = ppls[i:i + window_size]
        if np.isnan(window).any():
            densities.append(np.zeros_like(xs))  # pad with zeros for alignment
        else:
            kde = gaussian_kde(window, bw_method=bandwidth)
            densities.append(kde(xs))

    densities = np.array(densities)

    # Compute actual token indices of turn ends
    turn_boundaries_token_idx = np.cumsum([len(tok) for tok in token_list])[:-1]
    x_bins = np.arange(0, len(ppls) - window_size, step)
    turn_boundaries_bins = [np.searchsorted(x_bins, tb) for tb in turn_boundaries_token_idx]
    unique_turn_bins = sorted(set(turn_boundaries_bins))

    # Plot
    plt.figure(figsize=(12, 5))
    ax = sns.heatmap(
        densities.T,
        xticklabels=step,
        yticklabels=False,
        cmap="viridis",
        cbar=True,
        vmin=vmin,
        vmax=vmax
    )

    for xb in unique_turn_bins:
        ax.axvline(x=xb, color='white', linestyle='--', linewidth=0.5, alpha=0.7)

    plt.title(f"Rolling KDE Heatmap of PPL for {title}")
    plt.xlabel("Turn Window")
    plt.ylabel("Perplexity Bins")
    plt.tight_layout()
    plt.show()

    return densities


from matplotlib import cm
from matplotlib.colors import Normalize
from IPython.display import HTML


def display_tokens_colored_by_kde(
    token_list,
    ppls,
    densities,
    window_size,
    step,
    tokenizer,
    vmin=None,
    vmax=None,
    cmap_name="viridis"
):
    """
    Colors each token using the rolling KDE heatmap.
    
    Args:
        token_list: List of token tensors per turn.
        ppls: List of token-level perplexities (flattened).
        densities: 2D array of KDE values (n_windows x n_bins).
        window_size: Size of rolling window used for KDE.
        step: Step size used in rolling KDE.
        tokenizer: HF tokenizer.
        vmin, vmax: KDE value scale for coloring.
        cmap_name: Name of matplotlib colormap.

    Returns:
        IPython HTML display of color-coded tokens.
    """
    num_tokens = len(ppls)
    num_windows = densities.shape[0]
    token_kde_scores = np.zeros(num_tokens)
    token_counts = np.zeros(num_tokens)

    # For each window, distribute score across involved tokens
    for win_idx in range(num_windows):
        start = win_idx * step
        end = min(start + window_size, num_tokens)
        # Use max density for that window
        window_density = np.max(densities[win_idx])
        for i in range(start, end):
            token_kde_scores[i] += window_density
            token_counts[i] += 1

    # Normalize per token
    with np.errstate(divide='ignore', invalid='ignore'):
        averaged_scores = np.divide(token_kde_scores, token_counts, out=np.zeros_like(token_kde_scores), where=token_counts != 0)

    # Normalize colors
    norm = Normalize(vmin=vmin if vmin is not None else np.nanmin(averaged_scores),
                     vmax=vmax if vmax is not None else np.nanmax(averaged_scores))
    cmap = cm.get_cmap(cmap_name)

    # Flatten and decode tokens
    flat_tokens = [tok.item() for turn in token_list for tok in turn]
    decoded_tokens = tokenizer.convert_ids_to_tokens(flat_tokens)

    # HTML generation
    html = "<div style='font-family: monospace; line-height: 2;'>"
    for token, score in zip(decoded_tokens, averaged_scores):
        color = cm.colors.rgb2hex(cmap(norm(score)))
        clean_token = token.replace("Ġ", " ").replace("▁", " ").strip()
        html += f"<span style='background-color:{color}; padding:2px 4px; margin:1px; border-radius:3px;'>{clean_token}</span> "
    html += "</div>"

    return HTML(html)


In [7]:
import pandas as pd
import re
import numpy as np

candor_df = pd.read_csv("/u/sebono/conversational_dominance/data/processed/CANDOR_gpt2-large/conversations.csv")
candor_df.head()

,file_name,file_content
0,7595d874-c1cb-4aef-918f-3c81f3a84324,<SPK0> this is Yeah. Mhm. Mhm. Yeah. Yeah. Mot...
1,66768810-879d-4e2f-b5f5-ecc67d298104,<SPK0> Uh huh uh huh. Yeah. <SPK1> Yeah. All r...
2,dc3d0ae4-8d20-4d1d-8e0c-82c85a79b26e,<SPK1> Yeah. Right. Mhm. Mhm. Mhm. Okay. Yeah....
3,fb50ccf7-1a5a-4d80-b4f0-d5d1a1fd1135,<SPK0> Hi there and move away. <SPK1> mm. <SPK...
4,73c201ef-8c32-43d4-b642-dcedf4aa5c7f,<SPK0> Yeah. Okay. Yeah. Mhm. Yeah. Mhm Uh huh...


In [8]:
name_file = candor_df.iloc[0]["file_name"]
conversation = candor_df.iloc[0]["file_content"]

In [9]:
import re 
pattern = r'<(?:SPK[0-9]|MOD)>'
dialog_lines = conversation.replace(" <", "\n<").split("\n")[1:]
start_of_sentence=" "
matches = re.findall(pattern, "".join(dialog_lines))
token_list = [tokenizer(token, return_tensors="pt").input_ids[0] for token in dialog_lines]
encodings = tokenizer(f"{start_of_sentence}".join(dialog_lines), return_tensors="pt")

Token indices sequence length is longer than the specified maximum sequence length for this model (11410 > 1024). Running this sequence through the model will result in indexing errors


In [10]:
all_data = pd.read_csv(f"../information_exchange_labelling/{candor_df.iloc[0]['file_name']}_candor_ppl.csv")

In [11]:
assert len(all_data['baselined_p1_scores']) == len(encodings.input_ids[0])

In [12]:
all_data.keys()

Index(['baselined_p1_scores', 'baselined_p2_scores', 'baselined_p3_scores',
       'original_dialog_p1', 'original_dialog_p2', 'original_dialog_p3'],
      dtype='object')

In [ ]:
baselined_ppls_p1_spk = compute_dominance_per_spk(all_data["baselined_p1_scores"])
baselined_ppls_p2_spk = compute_dominance_per_spk(all_data["baselined_p2_scores"])
baselined_ppls_p3_spk = compute_dominance_per_spk(all_data["baselined_p3_scores"])

In [13]:
# Run statistical tests
from scipy import stats
def compute_significance(original_ppls_p_spk):
    spk1_clean = np.array(original_ppls_p_spk["<SPK0>"])
    spk2_clean = np.array(original_ppls_p_spk["<SPK1>"])

    ppl_spk1 = spk1_clean[~np.isnan(spk1_clean)]
    ppl_spk2 = spk2_clean[~np.isnan(spk2_clean)]

    ks_stat, ks_pvalue = stats.ks_2samp(ppl_spk1, ppl_spk2)
    mw_stat, mw_pvalue = stats.mannwhitneyu(ppl_spk1, ppl_spk2, alternative='two-sided')
    tt_stat, tt_pvalue = stats.ttest_ind(ppl_spk1, ppl_spk2, equal_var=False)

    return {
        "Kolmogorov-Smirnov": {"statistic": ks_stat, "p_value": ks_pvalue},
        "Mann-Whitney U": {"statistic": mw_stat, "p_value": mw_pvalue},
        "T-test": {"statistic": tt_stat, "p_value": tt_pvalue}
    }

In [14]:
compute_significance(baselined_ppls_p1_spk), compute_significance(baselined_ppls_p2_spk) , compute_significance(baselined_ppls_p3_spk) 

NameError: name 'baselined_ppls_p1_spk' is not defined

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.kdeplot(baselined_ppls_p2_spk["<SPK0>"], label="spk0", fill=True)
sns.kdeplot(baselined_ppls_p2_spk["<SPK1>"], label="spk1", fill=True)
plt.title("PPL Distributions by Speaker")
plt.legend()
plt.show()

In [ ]:
num_turns = 100
upper_limit = np.cumsum([len(token) for token in token_list])[num_turns]

In [ ]:
all_data.keys()

In [ ]:
densities = rolling_kde_heatmap_with_turns(token_list=token_list[:num_turns], ppls=all_data["original_dialog_p2"][:upper_limit],vmin=0, vmax=1.0)

In [ ]:
html = display_tokens_colored_by_kde(
    token_list=token_list[:num_turns],
    ppls=all_data["original_dialog_p2"][:upper_limit],
    densities=densities,  # output from rolling KDE
    window_size=50,
    step=10,
    tokenizer=tokenizer,
    vmin=0,
    vmax=1.0
)
display(html)

In [ ]:
tokenizer.decode(encodings.input_ids[0][200:250],skip_special_tokens=True)

In [ ]:
tokenizer.decode(encodings.input_ids[0][4200:4300],skip_special_tokens=True)

In [ ]:
all_data

In [ ]:
# BEFORE
col_1 = ["original_dialog_p1","original_dialog_p2","original_dialog_p3"]
baselined_p2_scores = all_data["original_dialog_p2"]
baselined_p3_scores = all_data["original_dialog_p3"]
baselined_p1_scores = all_data["original_dialog_p1"]
corr, fig_corr, p, fig_p, fig_r = correlation_heatmap(col_1,col_1,all_data)

In [ ]:
fig_corr

In [ ]:
all_data.bfill(inplace=True)

In [ ]:
from transformers import AutoTokenizer
import numpy as np

def remove_match_prefix_ppl(token_list, ppl, matches, tokenizer, min_len=3):
    """
    Removes matching prefixes from each line and filters out entire turns if remaining token length is below threshold.

    Args:
        token_list (List[List[int]]): Tokenized dialog lines.
        ppl (List[float]): Flat list of token-level PPL values.
        matches (List[str]): List of string prefixes to remove from each line.
        tokenizer: HuggingFace tokenizer used.
        min_len (int): Minimum length (after prefix removal) to keep the turn.

    Returns:
        Tuple[
            List[List[int]],  # filtered_tokens
            List[float],      # filtered_ppl
            List[int],        # flat_token_ids
            List[str]         # filtered_matches
        ]
    """
    assert len(token_list) == len(matches), "Mismatch between token list and matches"

    line_offsets = np.cumsum([0] + [len(t) for t in token_list[:-1]])  # start index in flat PPL array

    filtered_tokens = []
    filtered_ppl = []
    flat_token_ids = []
    filtered_matches = []

    for i, (token, match) in enumerate(zip(token_list, matches)):
        tok_match = tokenizer(match, return_tensors="pt")
        match_tok_len = len(tok_match.input_ids[0])

        # Slice off the match prefix
        token_filtered = token[match_tok_len:]

        if len(token_filtered) < min_len:
            continue  # skip turn entirely if below threshold

        # Keep this turn
        filtered_tokens.append(token_filtered)
        filtered_matches.append(match)

        # Extract aligned PPL values
        start = line_offsets[i] + match_tok_len
        end = line_offsets[i] + len(token)
        turn_ppl = ppl[start:end]

        assert len(turn_ppl) == len(token_filtered), f"PPL/token mismatch at turn {i}"
        
        filtered_ppl.extend(turn_ppl)
        flat_token_ids.extend(token_filtered)

    return filtered_tokens, filtered_ppl, flat_token_ids, filtered_matches

In [ ]:
all_data.bfill(inplace=True)

In [ ]:
filtered_tokens_p1, filtered_ppl_p1, filtered_encodings_p1, filtered_matches_p1 = remove_match_prefix_ppl(token_list, all_data['original_dialog_p1'], matches, tokenizer)
assert np.cumsum([len(token) for token in filtered_tokens_p1])[-1], len(filtered_ppl)

In [ ]:
filtered_tokens_p2, filtered_ppl_p2, filtered_encodings_p2 = remove_match_prefix_ppl(token_list, all_data['original_dialog_p2'], matches, tokenizer)
assert np.cumsum([len(token) for token in filtered_tokens_p2])[-1], len(filtered_ppl)

In [ ]:
filtered_tokens_p3, filtered_ppl_p3, filtered_encodings_p3 = remove_match_prefix_ppl(token_list, all_data['original_dialog_p3'], matches, tokenizer)
assert np.cumsum([len(token) for token in filtered_tokens_p3])[-1], len(filtered_ppl)

In [ ]:
assert len(filtered_tokens_p1) == len(filtered_tokens_p2)
assert len(filtered_tokens_p1) == len(filtered_tokens_p3)

In [ ]:
assert filtered_encodings_p1 == filtered_encodings_p2
assert filtered_encodings_p1 == filtered_encodings_p3
filtered_encodings = filtered_encodings_p1

In [ ]:
from sklearn.decomposition import PCA

P1 = np.asarray(filtered_ppl_p1)
P2 = np.asarray(filtered_ppl_p2)
P3 = np.asarray(filtered_ppl_p3)

X = np.column_stack([P1, P2, P3])
pca = PCA(n_components=3)
X_orthogonal = pca.fit_transform(X)

dim1_pca = X_orthogonal[:,0]
dim2_pca = X_orthogonal[:,1]
dim3_pca = X_orthogonal[:,2]
all_data_pca = pd.DataFrame({"dim1_pca": dim1_pca, "dim2_pca": dim2_pca, "dim3_pca":dim3_pca})

In [ ]:
col_1 = ["dim1_pca","dim2_pca","dim3_pca"]
corr, fig_corr, p, fig_p, fig_r = correlation_heatmap(col_1,col_1,all_data_pca)

In [ ]:
fig_corr

In [ ]:
import os
os.chdir("/u/sebono/conversational_dominance/information_exchange_labelling")
os.getcwd()

In [ ]:
from utils import rolling_kde_heatmap_with_turns

In [ ]:
densities_dim2_pca = rolling_kde_heatmap_with_turns(token_list=token_list, ppls=all_data_pca["dim2_pca"],vmin=0, vmax=1.0)

In [ ]:
all_data_pca["dim2_pca"]

In [ ]:
avg_density_dim2_pca = [densities_dim2_pca[t].mean() for t in range(len(densities_dim2_pca))]

In [ ]:
densities_dim3_pca = rolling_kde_heatmap_with_turns(token_list=token_list, ppls=all_data_pca["dim3_pca"],vmin=0, vmax=1.0)

In [ ]:
avg_density_dim3_pca = [densities_dim3_pca[t].mean() for t in range(len(densities_dim3_pca))]

In [ ]:
#KMeans (assumes spherical clusters):
from sklearn.cluster import KMeans

kmeans = KMeans(n_clusters=2, random_state=0)
labels = kmeans.fit_predict(X_orthogonal)

In [ ]:
import matplotlib.pyplot as plt
plt.scatter(X_orthogonal[:, 0], X_orthogonal[:, 1], c=labels, cmap="tab10", s=40)
plt.title("Clusters of Turns in P2/P3 PCA Space")
plt.xlabel("PC2")
plt.ylabel("PC3")
plt.colorbar(label="Cluster Label")
plt.grid(True)
plt.show()


In [ ]:
import numpy as np

# Step 1: get turn boundaries from dialog_lines
turn_lengths = [len(t) for t in filtered_tokens_p1]
turn_start = np.cumsum([0] + turn_lengths[:-1])
turn_end = np.cumsum(turn_lengths)

for cluster_id in np.unique(labels):
    print(f"\n--- Cluster {cluster_id} ---")
    cluster_token_indices = set(np.where(labels == cluster_id)[0])

    for start, end in zip(turn_start, turn_end):
        turn_indices = list(range(start, end))
        selected = [i for i in turn_indices if i in cluster_token_indices]
        if selected:
            full_sentence = tokenizer.decode([filtered_encodings[i] for i in turn_indices], skip_special_tokens=True)
            clustered_text = tokenizer.decode([filtered_encodings[i] for i in selected], skip_special_tokens=True)
            print(f"clustered_text: {clustered_text}\nfull_sentence:{full_sentence}")

In [ ]:
#DBSCAN (finds dense clusters, doesn't require specifying k):
from sklearn.cluster import DBSCAN

db = DBSCAN(eps=0.3, min_samples=2)
labels = db.fit_predict(X_orthogonal)

In [ ]:
import matplotlib.pyplot as plt
plt.scatter(X_orthogonal[:, 0], X_orthogonal[:, 1], c=labels, cmap="tab10", s=40)
plt.title("Clusters of Turns in P2/P3 PCA Space")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.colorbar(label="Cluster Label")
plt.grid(True)
plt.show()

In [ ]:
#Gaussian Mixture Model (softer boundaries):
from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(n_components=2, random_state=0)
labels = gmm.fit_predict(X_orthogonal)

In [ ]:
import matplotlib.pyplot as plt
plt.scatter(X_orthogonal[:, 0], X_orthogonal[:, 1], c=labels, cmap="tab10", s=40)
plt.title("Clusters of Turns in P2/P3 PCA Space")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.colorbar(label="Cluster Label")
plt.grid(True)
plt.show()

### Labels

In [ ]:
pattern = r'<(?:SPK[0-9]|MOD)>'
dialog_lines = conversation.replace("<", "\n<").split("\n")[1:]
start_of_sentence=" "
matches = re.findall(pattern, "".join(dialog_lines))
token_list = [tokenizer(token, return_tensors="pt").input_ids[0].detach().numpy() for token in dialog_lines]
encodings = tokenizer(f"{start_of_sentence}".join(dialog_lines), return_tensors="pt")

In [ ]:
directory = "/u/sebono/conversational_dominance/data/external/CANDOR/transcript_audiophile/"
directory_file = directory + f"{name_file}.csv"
transcript_audiophile = pd.read_csv(directory_file).iloc[1:]
transcript_audiophile.head()

In [ ]:
path_annotations = "/u/sebono/conversational_dominance/data/processed/CANDOR_gpt2-large/affective_labels/"
annotation_file = path_annotations + f"{name_file}.csv"
affective_labels = pd.read_csv(annotation_file)
affective_labels.head()

In [ ]:
speakers = list(np.unique(affective_labels["user_id"]))

In [ ]:
assert len(transcript_audiophile['utterance']) == len(token_list)

In [ ]:
cols = ["prob_face_anger","prob_face_contempt","prob_face_disgust","prob_face_fear","prob_face_happiness","prob_face_neutral","prob_face_sadness","prob_face_surprise","smile"]

In [ ]:
transcript_audiophile["tokens"] = token_list

In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict

def assign_words_to_bins(df, bin_size=1.0):
    """
    Assigns words to time bins based on uniform spread over the utterance duration.
    Ensures all bins from time 0 to max stop are represented.

    Returns:
        pd.DataFrame: each bin with its start time, assigned words, and count.
    """
    bin_tok = defaultdict(list)
    bin_ppl = defaultdict(list)
    tok_len = 0
    
    for _, row in df.iterrows():
        start = row["start"]
        stop = row["stop"]
        duration = stop - start

        if duration <= 0 or pd.isna(start) or pd.isna(stop):
            continue

        tokens = row["tokens"]
        n_tokens = len(tokens)
        
        if n_tokens == 0:
            continue

        # Compute per-word time positions assuming uniform spread
        tok_times = np.linspace(start, stop, n_tokens + 1)
        for i, tok in enumerate(tokens):
            tok_start = tok_times[i]
            tok_end = tok_times[i + 1]

            bin_start_idx = int(np.floor(tok_start / bin_size))
            bin_end_idx = int(np.floor(tok_end / bin_size))

            for b in range(bin_start_idx, bin_end_idx + 1):
                bin_tok[b].append(tok)
                bin_ppl[b].append(tok_len + i)

        tok_len += n_tokens
    # Determine full range of bins
    max_bin = int(np.ceil(df["stop"].max() / bin_size))
    all_bins = list(range(max_bin + 1))

    # Construct DataFrame with all bins, filling empty ones with []
    bins_df = pd.DataFrame([
        {
            "time_bin": b,
            "start_time": b * bin_size,
            "end_time": (b + 1) * bin_size,
            "words": tokenizer.decode(torch.tensor(bin_tok[b]), skip_special_tokens=True),
            "ppl": bin_ppl[b],
            "n_tok": len(bin_tok[b])
        }
        for b in all_bins
    ])

    return bins_df

In [ ]:
affective_labels["timedelta"] = pd.to_timedelta(affective_labels["timedelta"])
affective_labels["time_bin"] = affective_labels["timedelta"].dt.total_seconds().astype(int)

In [ ]:
affective_labels_spk1 = affective_labels[affective_labels['user_id']==speakers[0]][cols].reset_index()
affective_labels_spk2 = affective_labels[affective_labels['user_id']==speakers[1]][cols].reset_index()
assert affective_labels_spk2.shape == affective_labels_spk1.shape

In [ ]:
bins_df = assign_words_to_bins(transcript_audiophile)

In [ ]:
combined_df = pd.concat([bins_df.reset_index(drop=True), affective_labels_spk1.iloc[:bins_df.shape[0]].reset_index(drop=True)], axis=1)
combined_df

In [ ]:
import pandas as pd

def expand_multiple_ppl_by_token(bin_df, ppl_dict, annotation_cols=None):
    """
    Expands a bin-level DataFrame into token-level rows, adding multiple PPL values per token.

    Args:
        bin_df (pd.DataFrame): must contain 'ppl' as a list of token indices.
        ppl_dict (dict): dictionary of {name: List[float]}, e.g. {'ppl1': [...], 'ppl2': [...]}
        annotation_cols (List[str]): optional list of annotation columns to repeat per token.

    Returns:
        pd.DataFrame: one row per token with multiple ppl values and repeated annotations.
    """
    # Validate that all ppl lists have the same length
    lengths = [len(v) for v in ppl_dict.values()]
    if not all(l == lengths[0] for l in lengths):
        raise ValueError("All PPL arrays must have the same length.")

    records = []

    for _, row in bin_df.iterrows():
        token_ids = row["ppl"]
        if not isinstance(token_ids, list) or len(token_ids) == 0:
            continue

        for tok_id in token_ids:
            if tok_id >= lengths[0]:
                continue  # skip out-of-bounds

            record = {"token_id": tok_id}
            for name, ppl_values in ppl_dict.items():
                record[name] = ppl_values[tok_id]

            if annotation_cols:
                for col in annotation_cols:
                    record[col] = row.get(col, None)

            records.append(record)

    return pd.DataFrame(records)

In [ ]:
all_data_pca

In [ ]:
ppl_dict = {'dim2_pca': all_data_pca["dim2_pca"], 'dim3_pca': all_data_pca["dim3_pca"]}
all_data = expand_multiple_ppl_by_token(combined_df, ppl_dict, cols)
all_data.head()

In [ ]:
# BEFORE
col_2 = ["prob_face_anger", "prob_face_contempt", "prob_face_disgust", "prob_face_fear", "prob_face_happiness", "prob_face_neutral", "prob_face_sadness", "prob_face_surprise", "smile"]
col_1 = ["dim2_pca","dim3_pca"]

corr, fig_corr, p, fig_p, fig_r = correlation_heatmap(col_1,col_2,all_data)

In [ ]:
fig_corr

In [ ]:
fig_r

In [ ]:
from scipy.stats import spearmanr
all_data.dropna(inplace=True)
spearmanr(all_data['dim2_pca'], all_data[cols])

In [ ]:
from scipy.stats import spearmanr
import pandas as pd

# Perform correlation
stat, pval = spearmanr(all_data['dim2_pca'], all_data[cols], nan_policy='omit')

# Extract just the relevant row (first row after the scalar)
correlation_vector = stat[0, 1:]
pval_vector = pval[0, 1:]

# Wrap into a DataFrame
corr_df = pd.DataFrame({
    "feature": cols,
    "spearman_r": correlation_vector,
    "p_value": pval_vector
})

# Sort by absolute correlation
corr_df["abs_r"] = corr_df["spearman_r"].abs()
corr_df = corr_df.sort_values("abs_r", ascending=False)

corr_df[["feature", "spearman_r", "p_value"]]


In [ ]:
all_data

In [ ]:
from sklearn.feature_selection import mutual_info_regression

# X: affective features, y: P2_pca
X = all_data[cols].fillna(0)  # Replace NaNs or handle appropriately
y = all_data['dim3_pca']

mi_scores = mutual_info_regression(X, y, random_state=42)

mi_df = pd.DataFrame({
    "feature": cols,
    "mutual_info": mi_scores
}).sort_values("mutual_info", ascending=False)

print(mi_df)


In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X, y)

rf_importances = pd.DataFrame({
    "feature": cols,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

print(rf_importances)


In [ ]:
from sklearn.inspection import partial_dependence, PartialDependenceDisplay

PartialDependenceDisplay.from_estimator(rf, X, ['prob_face_surprise'])
plt.show()


In [ ]:
features = ["prob_face_neutral", "prob_face_sadness", "prob_face_disgust", "prob_face_fear", "prob_face_anger", "prob_face_happiness", "prob_face_contempt", "prob_face_surprise" ]

In [ ]:
# Re-import necessary libraries after kernel reset
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Simulate example data structure for the plot (replace with actual data in real usage)
# This is just to reinitialize after kernel reset
# Replace this with loading actual 'all_data' if available
import numpy as np
np.random.seed(0)

# Scatterplot of prob_face_anger vs P2_pca
plt.figure(figsize=(8, 5))
sns.scatterplot(x=all_data["prob_face_surprise"], y=all_data["dim2_pca"], alpha=0.3)
plt.xlabel("Probability of Face Anger")
plt.ylabel("P2 PCA Component")
plt.title("Relationship between Face Anger and P2_pca")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import statsmodels.api as sm
x = all_data["prob_face_surprise"]
y = all_data["dim2_pca"]
lowess = sm.nonparametric.lowess
z = lowess(y, x, frac=0.2)
plt.plot(z[:, 0], z[:, 1], color='red')

In [ ]:
x = all_data["prob_face_surprise"]
y = all_data["dim2_pca"]
plt.hexbin(x, y, gridsize=50, cmap="Purples")
plt.colorbar(label="Density")


In [ ]:
binned = pd.cut(all_data["prob_face_surprise"], bins=10)
means = all_data.groupby(binned)["dim2_pca"].mean()

# Plot
means.plot(marker='o', linestyle='-')
plt.xlabel("Binned prob_face_anger")
plt.ylabel("Mean P2_pca")
plt.title("Mean P2_pca by Face Anger Bins")
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
binned = pd.cut(all_data["prob_face_anger"], bins=10)
grouped = all_data.groupby(binned)["dim2_pca"].mean()
grouped.plot(kind="bar")


In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

# Features and target
X = all_data[["prob_face_anger"]]  # or multiple features
y = all_data["dim2_pca"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Fit tree
tree = DecisionTreeRegressor(max_depth=4, random_state=42)
tree.fit(X_train, y_train)

# Predict and evaluate
y_pred = tree.predict(X_test)
print("R^2:", r2_score(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))

# Plot the fit
import matplotlib.pyplot as plt
xx = np.linspace(X.min()[0], X.max()[0], 500).reshape(-1, 1)
yy = tree.predict(xx)
plt.scatter(X, y, alpha=0.3)
plt.plot(xx, yy, color="red", lw=2)
plt.xlabel("prob_face_anger")
plt.ylabel("P2_pca")
plt.title("Decision Tree Fit")
plt.show()


In [ ]:
import statsmodels.api as sm
from patsy import dmatrix

# Generate spline basis for prob_face_anger
x_spline = dmatrix("bs(prob_face_anger, df=5, degree=3, include_intercept=False)", 
                   data=all_data, return_type='dataframe')

# Fit model
y = all_data["dim2_pca"]
model = sm.OLS(y, x_spline).fit()

# Predict
x_pred = np.linspace(all_data["prob_face_anger"].min(), all_data["prob_face_anger"].max(), 300)
x_pred_spline = dmatrix("bs(x, df=5, degree=3, include_intercept=False)", 
                        {"x": x_pred}, return_type='dataframe')
y_pred = model.predict(x_pred_spline)

# Plot
plt.scatter(all_data["prob_face_anger"], y, facecolors='none', edgecolors='b', alpha=0.3)
plt.plot(x_pred, y_pred, color="red")
plt.xlabel("prob_face_anger")
plt.ylabel("P2_pca")
plt.title("Spline Regression Fit")
plt.show()


In [ ]:
# Re-import necessary modules after kernel reset
import pandas as pd
import numpy as np
from sklearn.preprocessing import SplineTransformer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt

# Simulate minimal structure (replace this with actual `all_data`)
np.random.seed(0)
N = 500

# Features for GAM-like spline model
features = ["prob_face_anger", "prob_face_happiness"]
X = all_data[features]
y = all_data["dim2_pca"]

# Multivariate spline regression via sklearn pipeline
spline_model = make_pipeline(
    SplineTransformer(degree=3, n_knots=5, include_bias=False),
    LinearRegression()
)
spline_model.fit(X, y)
spline_preds = spline_model.predict(X)

# Plot predictions vs true
plt.figure(figsize=(6, 4))
plt.scatter(y, spline_preds, alpha=0.5)
plt.xlabel("True P2_pca")
plt.ylabel("Spline Model Prediction")
plt.title("Multivariate Spline Regression")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
all_data

In [ ]:
# Features and target
X = all_data[[
    "prob_face_anger", "prob_face_happiness", "prob_face_sadness",
    "prob_face_fear", "prob_face_disgust", "prob_face_surprise", "smile"
]]
y = all_data["dim2_pca"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
from pygam import LinearGAM, s

gam = LinearGAM(s(0) + s(1) + s(2) + s(3) + s(4) + s(5) + s(6)).fit(X_train, y_train)
y_pred = gam.predict(X_test)

print("GAM R²:", r2_score(y_test, y_pred))

In [ ]:
import shap
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor().fit(X_train, y_train)
explainer = shap.Explainer(rf, X_train)
shap_values = explainer(X_test)
shap.plots.beeswarm(shap_values)

In [ ]:
# Features and target
X = all_data[[
    "prob_face_anger", "prob_face_happiness", "prob_face_sadness",
    "prob_face_fear", "prob_face_disgust", "prob_face_surprise", "smile"
]]
y = all_data["dim3_pca"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
from pygam import LinearGAM, s

gam = LinearGAM(s(0) + s(1) + s(2) + s(3) + s(4) + s(5) + s(6)).fit(X_train, y_train)
y_pred = gam.predict(X_test)

print("GAM R²:", r2_score(y_test, y_pred))

In [ ]:
import shap
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor().fit(X_train, y_train)
explainer = shap.Explainer(rf, X_train)
shap_values = explainer(X_test)
shap.plots.beeswarm(shap_values)